In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
import logging

logging.basicConfig(level=logging.INFO)

# === Paths ===
BASE_DIR      = Path(r"D:\Shikha\bias_project")
WORD_LISTS    = BASE_DIR / "word_lists" / "caliskan_2017"
CORPUS_DIR    = BASE_DIR / "corpus"
OUTPUT_DIR    = BASE_DIR / "outputs" / "embeddings" / "word2vec"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === Load word lists for WEAT 6 (Career vs Family) ===
# This is the most relevant test for labor force participation

def load_word_list(filepath):
    with open(filepath, 'r') as f:
        return [w.strip().lower() for w in f.readlines() if w.strip()]

# WEAT 6 — target and attribute words
career_terms  = load_word_list(WORD_LISTS / "career_terms_weat6.txt")
family_terms  = load_word_list(WORD_LISTS / "family_terms_weat6.txt")
male_names    = load_word_list(WORD_LISTS / "male_names_weat6.txt")
female_names  = load_word_list(WORD_LISTS / "female_names_weat6.txt")

print("=== WEAT 6 Word Lists ===")
print(f"Career terms  : {career_terms}")
print(f"Family terms  : {family_terms}")
print(f"Male names    : {male_names}")
print(f"Female names  : {female_names}")

=== WEAT 6 Word Lists ===
Career terms  : ['executive', 'management', 'professional', 'corporation', 'salary', 'office', 'business', 'career']
Family terms  : ['home', 'parents', 'children', 'family', 'cousins', 'marriage', 'wedding', 'relatives']
Male names    : ['john', 'paul', 'mike', 'kevin', 'steve', 'greg', 'jeff', 'bill']
Female names  : ['amy', 'joan', 'lisa', 'sarah', 'diana', 'kate', 'ann', 'donna']


In [3]:
# Load all corpus sources and combine into per-state text
# We'll train one Word2Vec model per state

def load_congressional_record():
    path = CORPUS_DIR / "congressional_record" / "2024-01" / "speeches.csv"
    df = pd.read_csv(path)
    print(f"Congressional Record: {len(df)} speeches, columns: {list(df.columns)}")
    return df

def load_chronicling_america():
    path = CORPUS_DIR / "chronicling_america" / "1920-1922" / "articles.csv"
    df = pd.read_csv(path)
    print(f"Chronicling America: {len(df)} pages, columns: {list(df.columns)}")
    return df

def load_gdelt():
    path = CORPUS_DIR / "gdelt" / "2024-01" / "gdelt_articles_with_text.csv"
    df = pd.read_csv(path)
    # Keep only successfully fetched articles
    df = df[df['text'].notna() & (df['text'] != '')]
    print(f"GDELT: {len(df)} articles, columns: {list(df.columns)}")
    return df

def load_reddit():
    posts    = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "posts.csv")
    comments = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "comments.csv")
    print(f"Reddit: {len(posts)} posts, {len(comments)} comments, columns: {list(posts.columns)}")
    return posts, comments

# Load all sources
print("Loading corpora...\n")
cr_df   = load_congressional_record()
ca_df   = load_chronicling_america()
gd_df   = load_gdelt()
rd_posts, rd_comments = load_reddit()

Loading corpora...

Congressional Record: 547 speeches, columns: ['date', 'package_id', 'granule_id', 'granule_class', 'title', 'speaker_name', 'state', 'party', 'chamber', 'bioguide_id', 'speaker_chamber', 'text', 'text_length']
Chronicling America: 1350 pages, columns: ['state', 'state_abbr', 'year', 'item_id', 'title', 'date', 'text_length', 'text']
GDELT: 702 articles, columns: ['article_url', 'state', 'source_domain', 'source_name', 'source_size', 'article_date', 'fetch_status', 'text_length', 'text']
Reddit: 4423 posts, 4563 comments, columns: ['state', 'state_abbr', 'subreddit', 'post_id', 'title', 'selftext', 'score', 'created_utc', 'text_length']


In [4]:
# Combine all sources into a single per-state text dictionary
# state_abbr -> list of tokenized sentences

from gensim.utils import simple_preprocess

def tokenize(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []
    return simple_preprocess(text, deacc=True, min_len=2)

# Build state -> sentences mapping
state_corpus = {}  # state_abbr -> list of token lists

# 1. Congressional Record (state column has full state name, need abbr)
# Check what format state is in
print("CR state sample:", cr_df['state'].dropna().unique()[:5])

# 2. Chronicling America
print("CA state sample:", ca_df['state_abbr'].dropna().unique()[:5])

# 3. GDELT
print("GDELT state sample:", gd_df['state'].dropna().unique()[:5])

# 4. Reddit
print("Reddit state sample:", rd_posts['state_abbr'].dropna().unique()[:5])

CR state sample: <ArrowStringArray>
['NY', 'KY', 'IL', 'NC', 'IA']
Length: 5, dtype: str
CA state sample: <ArrowStringArray>
['OH', 'WY', 'CA', 'TX', 'IL']
Length: 5, dtype: str
GDELT state sample: <ArrowStringArray>
['CO', 'PA', 'WA', 'ND', 'IA']
Length: 5, dtype: str
Reddit state sample: <ArrowStringArray>
['AL', 'AK', 'AZ', 'AR', 'CA']
Length: 5, dtype: str


In [5]:
# Build per-(state, period) corpus — proposal-correct design
# Periods: hist_1920s (Chronicling America), modern_2020 (Reddit), modern_2024 (CR + GDELT)

state_period_corpus = {}  # (state_abbr, period) -> list of token lists

def add_to_corpus(state_abbr, period, text):
    if not isinstance(text, str) or len(text.strip()) < 50:
        return
    tokens = tokenize(text)
    if len(tokens) < 5:
        return
    key = (state_abbr, period)
    if key not in state_period_corpus:
        state_period_corpus[key] = []
    state_period_corpus[key].append(tokens)

# 1. Chronicling America → hist_1920s
for _, row in ca_df.iterrows():
    add_to_corpus(str(row['state_abbr']), 'hist_1920s', str(row.get('text', '')))

# 2. Reddit → modern_2020
for _, row in rd_posts.iterrows():
    text = str(row.get('title', '')) + ' ' + str(row.get('selftext', ''))
    add_to_corpus(str(row['state_abbr']), 'modern_2020', text)
for _, row in rd_comments.iterrows():
    add_to_corpus(str(row['state_abbr']), 'modern_2020', str(row.get('body', '')))

# 3. Congressional Record + GDELT → modern_2024
for _, row in cr_df.iterrows():
    add_to_corpus(str(row['state']), 'modern_2024', str(row.get('text', '')))
for _, row in gd_df.iterrows():
    add_to_corpus(str(row['state']), 'modern_2024', str(row.get('text', '')))

# Summary
periods = ['hist_1920s', 'modern_2020', 'modern_2024']
print(f"{'Period':<14} {'State cells':>12} {'Total sentences':>16}")
print("-" * 45)
for p in periods:
    cells = [k for k in state_period_corpus if k[1] == p]
    sents = sum(len(state_period_corpus[k]) for k in cells)
    print(f"{p:<14} {len(cells):>12} {sents:>16,}")

Period          State cells  Total sentences
---------------------------------------------
hist_1920s                9            1,350
modern_2020              50            6,447
modern_2024              52            1,249


In [6]:
GARG_LISTS = BASE_DIR / "word_lists" / "garg_2018"
male_pairs   = load_word_list(GARG_LISTS / "male_pairs.txt")
female_pairs = load_word_list(GARG_LISTS / "female_pairs.txt")

print(f"Male pairs   ({len(male_pairs)}): {male_pairs}")
print(f"Female pairs ({len(female_pairs)}): {female_pairs}")

def cosine_sim(v1, v2):
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0.0
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def get_vector(model, word):
    try:
        return model.wv[word]
    except KeyError:
        return None

Male pairs   (20): ['he', 'son', 'his', 'him', 'father', 'man', 'boy', 'himself', 'male', 'brother', 'sons', 'fathers', 'men', 'boys', 'males', 'brothers', 'uncle', 'uncles', 'nephew', 'nephews']
Female pairs (20): ['she', 'daughter', 'hers', 'her', 'mother', 'woman', 'girl', 'herself', 'female', 'sister', 'daughters', 'mothers', 'women', 'girls', 'females', 'sisters', 'aunt', 'aunts', 'niece', 'nieces']


In [7]:
def weat_effect_size(model, X_words, Y_words, A_words, B_words,
                     min_targets=3, min_attrs=2):
    """Caliskan effect size with relaxed coverage + coverage reporting."""
    vecs_A = [get_vector(model, a) for a in A_words if get_vector(model, a) is not None]
    vecs_B = [get_vector(model, b) for b in B_words if get_vector(model, b) is not None]
    if len(vecs_A) < min_attrs or len(vecs_B) < min_attrs:
        return None, {}

    def s(word):
        w_vec = get_vector(model, word)
        if w_vec is None:
            return None
        return (np.mean([cosine_sim(w_vec, v) for v in vecs_A]) -
                np.mean([cosine_sim(w_vec, v) for v in vecs_B]))

    s_X = [x for x in (s(w) for w in X_words) if x is not None]
    s_Y = [y for y in (s(w) for w in Y_words) if y is not None]

    if len(s_X) < min_targets or len(s_Y) < min_targets:
        return None, {}

    all_s = s_X + s_Y
    std = np.std(all_s, ddof=1)
    if std == 0:
        return None, {}

    coverage = {
        'n_X': len(s_X), 'n_Y': len(s_Y),
        'n_A': len(vecs_A), 'n_B': len(vecs_B)
    }
    return (np.mean(s_X) - np.mean(s_Y)) / std, coverage

# === Multi-seed scoring with coverage reporting ===
SEEDS   = [42, 7, 123, 2024, 99]
EXCLUDE = {'GU', 'PR', 'VI', 'AS', 'MP'}

results = []

print(f"{'State':<7} {'Period':<13} {'Pairs d':>9} {'±std':>7} {'Coverage (M/F/Car/Fam)':>24}")
print("-" * 65)

for (state_abbr, period) in sorted(state_period_corpus.keys()):
    if state_abbr in EXCLUDE:
        continue
    sentences = state_period_corpus[(state_abbr, period)]
    if len(sentences) < 50:
        continue

    pairs_scores, cov = [], {}
    for seed in SEEDS:
        model = Word2Vec(
            sentences=sentences, vector_size=100, window=5,
            min_count=1, workers=1, epochs=20, seed=seed
        )
        d, c = weat_effect_size(model, male_pairs, female_pairs,
                                career_terms, family_terms)
        if d is not None:
            pairs_scores.append(d)
            cov = c

    if len(pairs_scores) >= 3:
        mean_d = np.mean(pairs_scores)
        std_d  = np.std(pairs_scores, ddof=1)
        cov_str = f"{cov['n_X']}/{cov['n_Y']}/{cov['n_A']}/{cov['n_B']}"
        print(f"{state_abbr:<7} {period:<13} {mean_d:>9.3f} {std_d:>7.3f} {cov_str:>24}")
        results.append({
            'state': state_abbr, 'period': period,
            'n_sentences': len(sentences),
            'weat6_pairs_d':   round(mean_d, 4),
            'weat6_pairs_std': round(std_d, 4),
            'cov_male': cov['n_X'], 'cov_female': cov['n_Y'],
            'cov_career': cov['n_A'], 'cov_family': cov['n_B'],
            'n_seeds': len(pairs_scores),
            'method': 'word2vec_pairs_effectsize'
        })

print(f"\nState-period cells scored: {len(results)}")

INFO:gensim.models.word2vec:collecting all words and their counts
INFO:gensim.models.word2vec:PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
INFO:gensim.models.word2vec:collected 1175 word types from a corpus of 3155 raw words and 100 sentences
INFO:gensim.models.word2vec:Creating a fresh vocabulary
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'effective_min_count=1 retains 1175 unique words (100.00% of original 1175, drops 0)', 'datetime': '2026-06-19T05:11:10.114916', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'prepare_vocab'}
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'effective_min_count=1 leaves 3155 word corpus (100.00% of original 3155, drops 0)', 'datetime': '2026-06-19T05:11:10.116963', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.

State   Period          Pairs d    ±std   Coverage (M/F/Car/Fam)
-----------------------------------------------------------------


INFO:gensim.models.word2vec:EPOCH 3: training on 3155 raw words (2462 effective words) took 0.0s, 311208 effective words/s
INFO:gensim.models.word2vec:EPOCH 4: training on 3155 raw words (2475 effective words) took 0.0s, 318291 effective words/s
INFO:gensim.models.word2vec:EPOCH 5: training on 3155 raw words (2470 effective words) took 0.0s, 281587 effective words/s
INFO:gensim.models.word2vec:EPOCH 6: training on 3155 raw words (2463 effective words) took 0.0s, 302833 effective words/s
INFO:gensim.models.word2vec:EPOCH 7: training on 3155 raw words (2458 effective words) took 0.0s, 319906 effective words/s
INFO:gensim.models.word2vec:EPOCH 8: training on 3155 raw words (2467 effective words) took 0.0s, 377315 effective words/s
INFO:gensim.models.word2vec:EPOCH 9: training on 3155 raw words (2488 effective words) took 0.0s, 301273 effective words/s
INFO:gensim.models.word2vec:EPOCH 10: training on 3155 raw words (2446 effective words) took 0.0s, 332626 effective words/s
INFO:gensim.mod

AL      modern_2020       0.090   0.291                  7/5/2/5


INFO:gensim.models.word2vec:EPOCH 8: training on 4987 raw words (3917 effective words) took 0.0s, 294595 effective words/s
INFO:gensim.models.word2vec:EPOCH 9: training on 4987 raw words (3935 effective words) took 0.0s, 345030 effective words/s
INFO:gensim.models.word2vec:EPOCH 10: training on 4987 raw words (3885 effective words) took 0.0s, 360600 effective words/s
INFO:gensim.models.word2vec:EPOCH 11: training on 4987 raw words (3930 effective words) took 0.0s, 344749 effective words/s
INFO:gensim.models.word2vec:EPOCH 12: training on 4987 raw words (3948 effective words) took 0.0s, 312379 effective words/s
INFO:gensim.models.word2vec:EPOCH 13: training on 4987 raw words (3914 effective words) took 0.0s, 469693 effective words/s
INFO:gensim.models.word2vec:EPOCH 14: training on 4987 raw words (3894 effective words) took 0.0s, 396849 effective words/s
INFO:gensim.models.word2vec:EPOCH 15: training on 4987 raw words (3911 effective words) took 0.0s, 448664 effective words/s
INFO:gensi

...

TN      modern_2020       0.189   1.157                  7/3/3/5


INFO:gensim.models.word2vec:EPOCH 2: training on 28419 raw words (22845 effective words) took 0.0s, 584787 effective words/s
INFO:gensim.models.word2vec:EPOCH 3: training on 28419 raw words (22854 effective words) took 0.1s, 452303 effective words/s
INFO:gensim.models.word2vec:EPOCH 4: training on 28419 raw words (22758 effective words) took 0.0s, 558684 effective words/s
INFO:gensim.models.word2vec:EPOCH 5: training on 28419 raw words (22799 effective words) took 0.0s, 582794 effective words/s
INFO:gensim.models.word2vec:EPOCH 6: training on 28419 raw words (22770 effective words) took 0.0s, 538561 effective words/s
INFO:gensim.models.word2vec:EPOCH 7: training on 28419 raw words (22839 effective words) took 0.0s, 494569 effective words/s
INFO:gensim.models.word2vec:EPOCH 8: training on 28419 raw words (22817 effective words) took 0.0s, 458834 effective words/s
INFO:gensim.models.word2vec:EPOCH 9: training on 28419 raw words (22815 effective words) took 0.1s, 371780 effective words/s


UT      modern_2020      -0.350   0.600                  5/4/3/3


INFO:gensim.models.word2vec:EPOCH 9: training on 5362 raw words (4299 effective words) took 0.0s, 407060 effective words/s
INFO:gensim.models.word2vec:EPOCH 10: training on 5362 raw words (4319 effective words) took 0.0s, 360124 effective words/s
INFO:gensim.models.word2vec:EPOCH 11: training on 5362 raw words (4232 effective words) took 0.0s, 220913 effective words/s
INFO:gensim.models.word2vec:EPOCH 12: training on 5362 raw words (4296 effective words) took 0.0s, 312948 effective words/s
INFO:gensim.models.word2vec:EPOCH 13: training on 5362 raw words (4266 effective words) took 0.0s, 299911 effective words/s
INFO:gensim.models.word2vec:EPOCH 14: training on 5362 raw words (4309 effective words) took 0.0s, 329017 effective words/s
INFO:gensim.models.word2vec:EPOCH 15: training on 5362 raw words (4307 effective words) took 0.0s, 327870 effective words/s
INFO:gensim.models.word2vec:EPOCH 16: training on 5362 raw words (4282 effective words) took 0.0s, 279508 effective words/s
INFO:gens

WY      modern_2020       0.133   1.119                  8/3/3/3

State-period cells scored: 38


In [8]:
# Diagnostic — check WEAT word coverage in a few representative cells
test_cells = [('WY', 'hist_1920s'), ('TX', 'modern_2020'), ('OH', 'modern_2024'), ('CA', 'modern_2020')]

for key in test_cells:
    if key not in state_period_corpus:
        print(f"{key}: NOT IN CORPUS")
        continue
    sentences = state_period_corpus[key]
    vocab = set(w for sent in sentences for w in sent)
    
    print(f"\n=== {key[0]} {key[1]} ({len(sentences)} sents, {len(vocab):,} unique words) ===")
    for name, words in [('career', career_terms), ('family', family_terms),
                        ('m_pairs', male_pairs), ('f_pairs', female_pairs),
                        ('m_names', male_names), ('f_names', female_names)]:
        present = [w for w in words if w in vocab]
        missing = [w for w in words if w not in vocab]
        print(f"  {name:<9}: {len(present)}/{len(words)} present | missing: {missing}")


=== WY hist_1920s (150 sents, 2,724 unique words) ===
  career   : 1/8 present | missing: ['executive', 'management', 'professional', 'corporation', 'salary', 'business', 'career']
  family   : 4/8 present | missing: ['parents', 'marriage', 'wedding', 'relatives']
  m_pairs  : 8/20 present | missing: ['father', 'boy', 'himself', 'male', 'brother', 'boys', 'males', 'brothers', 'uncle', 'uncles', 'nephew', 'nephews']
  f_pairs  : 7/20 present | missing: ['she', 'hers', 'girl', 'herself', 'female', 'sister', 'daughters', 'females', 'sisters', 'aunt', 'aunts', 'niece', 'nieces']
  m_names  : 2/8 present | missing: ['paul', 'mike', 'kevin', 'steve', 'greg', 'jeff']
  f_names  : 1/8 present | missing: ['joan', 'lisa', 'sarah', 'diana', 'kate', 'ann', 'donna']

=== TX modern_2020 (121 sents, 1,198 unique words) ===
  career   : 3/8 present | missing: ['executive', 'management', 'corporation', 'salary', 'career']
  family   : 1/8 present | missing: ['home', 'parents', 'children', 'cousins', '

In [9]:
import json

scores_df = pd.DataFrame(results).sort_values(['period', 'state'])

# Quality flag: reliable if seed-noise std < 0.25 and decent gender-word coverage
scores_df['reliable'] = (
    (scores_df['weat6_pairs_std'] < 0.25) &
    (scores_df['cov_male'] >= 5) & (scores_df['cov_female'] >= 4)
)

OUTPUT_SCORES = OUTPUT_DIR / "weat6_scores_word2vec_v2.csv"
scores_df.to_csv(OUTPUT_SCORES, index=False)

# === Gap documentation per period ===
ALL_STATES = set(STATE_TO_REGION := {
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN',
    'IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV',
    'NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN',
    'TX','UT','VT','VA','WA','WV','WI','WY'
})

gap_doc = {'method': 'word2vec_pairs_effectsize_multiseed', 'periods': {}}
for period in ['hist_1920s', 'modern_2020', 'modern_2024']:
    scored = set(scores_df[scores_df['period'] == period]['state'])
    # hist_1920s only covers the 9 extracted states
    expected = {'OH','WY','CA','TX','IL','MS','MT','OR','NY'} if period == 'hist_1920s' else ALL_STATES
    gap = sorted(expected - scored)
    reliable = set(scores_df[(scores_df['period']==period) & scores_df['reliable']]['state'])
    gap_doc['periods'][period] = {
        'expected_states': len(expected),
        'scored': len(scored),
        'reliable': len(reliable),
        'gap_states': gap,
        'gap_reason': 'Insufficient WEAT term coverage or <3 valid seeds at demo corpus scale'
    }

gap_doc['mitigation'] = 'Full-scale extraction will increase per-cell corpus 10-50x, resolving coverage gaps and reducing seed noise.'

GAP_FILE = OUTPUT_DIR / "weat6_gap_documentation_v2.json"
with open(GAP_FILE, 'w') as f:
    json.dump(gap_doc, f, indent=2)

# === Report ===
print(f"Total cells scored : {len(scores_df)}")
print(f"Reliable cells     : {scores_df['reliable'].sum()}")
print(f"\nPer period:")
for p, info in gap_doc['periods'].items():
    print(f"  {p:<13}: {info['scored']}/{info['expected_states']} scored, {info['reliable']} reliable, {len(info['gap_states'])} gap")

print(f"\nReliable cells detail:")
print(scores_df[scores_df['reliable']][
    ['state','period','weat6_pairs_d','weat6_pairs_std','n_sentences']
].to_string(index=False))

print(f"\nSaved to : {OUTPUT_SCORES}")
print(f"Gap doc  : {GAP_FILE}")

Total cells scored : 38
Reliable cells     : 8

Per period:
  hist_1920s   : 5/9 scored, 4 reliable, 4 gap
  modern_2020  : 30/50 scored, 1 reliable, 20 gap
  modern_2024  : 3/50 scored, 3 reliable, 47 gap

Reliable cells detail:
state      period  weat6_pairs_d  weat6_pairs_std  n_sentences
   CA  hist_1920s         0.1262           0.2383          150
   MS  hist_1920s        -0.5391           0.1964          150
   MT  hist_1920s        -1.1587           0.0965          150
   NY  hist_1920s         0.2163           0.1415          150
   MS modern_2020         0.3162           0.1981          124
   CA modern_2024         0.6400           0.1461           69
   IL modern_2024        -0.0343           0.1458           53
   NY modern_2024         0.2039           0.0907          131

Saved to : D:\Shikha\bias_project\outputs\embeddings\word2vec\weat6_scores_word2vec_v2.csv
Gap doc  : D:\Shikha\bias_project\outputs\embeddings\word2vec\weat6_gap_documentation_v2.json


In [10]:
# === Measure 2: Competence–Warmth | Measure 3: Leadership–Subordination ===
# Both use Garg gender pairs as targets, per proposal

# Load attribute lists
competence_terms = load_word_list(GARG_LISTS / "adjectives_intelligencegeneral.txt")
warmth_terms_raw = load_word_list(GARG_LISTS / "adjectives_sensitive.txt")

# Remove competence-overlapping words from warmth list (documented curation)
OVERLAP_REMOVED = [w for w in warmth_terms_raw if w in competence_terms]
warmth_terms = [w for w in warmth_terms_raw if w not in competence_terms]
print(f"Warmth list: removed {len(OVERLAP_REMOVED)} competence-overlapping words: {OVERLAP_REMOVED}")
print(f"Warmth list final: {len(warmth_terms)} words\n")

# Load custom leadership lists
LEAD_DIR = BASE_DIR / "word_lists" / "custom_leadership"
leadership_terms    = load_word_list(LEAD_DIR / "leadership_terms.txt")
subordination_terms = load_word_list(LEAD_DIR / "subordination_terms.txt")
print(f"Leadership terms    : {len(leadership_terms)}")
print(f"Subordination terms : {len(subordination_terms)}\n")

# === Score both measures across all cells ===
SEEDS = [42, 7, 123, 2024, 99]
results_m23 = []

print(f"{'State':<7} {'Period':<13} {'CompWarm d':>11} {'±std':>7} {'LeadSub d':>10} {'±std':>7}")
print("-" * 62)

for (state_abbr, period) in sorted(state_period_corpus.keys()):
    if state_abbr in EXCLUDE:
        continue
    sentences = state_period_corpus[(state_abbr, period)]
    if len(sentences) < 50:
        continue

    cw_scores, ls_scores = [], []
    for seed in SEEDS:
        model = Word2Vec(
            sentences=sentences, vector_size=100, window=5,
            min_count=1, workers=1, epochs=20, seed=seed
        )
        # Measure 2: positive = male more associated with competence
        d_cw, _ = weat_effect_size(model, male_pairs, female_pairs,
                                   competence_terms, warmth_terms)
        # Measure 3: positive = male more associated with leadership
        d_ls, _ = weat_effect_size(model, male_pairs, female_pairs,
                                   leadership_terms, subordination_terms)
        if d_cw is not None:
            cw_scores.append(d_cw)
        if d_ls is not None:
            ls_scores.append(d_ls)

    row = {'state': state_abbr, 'period': period, 'n_sentences': len(sentences)}
    cw_str, ls_str = "   --", "   --"
    
    if len(cw_scores) >= 3:
        row['compwarm_d']   = round(np.mean(cw_scores), 4)
        row['compwarm_std'] = round(np.std(cw_scores, ddof=1), 4)
        cw_str = f"{np.mean(cw_scores):>7.3f} {np.std(cw_scores, ddof=1):>7.3f}"
    if len(ls_scores) >= 3:
        row['leadsub_d']   = round(np.mean(ls_scores), 4)
        row['leadsub_std'] = round(np.std(ls_scores, ddof=1), 4)
        ls_str = f"{np.mean(ls_scores):>7.3f} {np.std(ls_scores, ddof=1):>7.3f}"
    
    if 'compwarm_d' in row or 'leadsub_d' in row:
        print(f"{state_abbr:<7} {period:<13} {cw_str:>19} {ls_str:>18}")
        results_m23.append(row)

print(f"\nCells scored (either measure): {len(results_m23)}")
print(f"Competence–Warmth cells : {sum(1 for r in results_m23 if 'compwarm_d' in r)}")
print(f"Leadership–Subord cells : {sum(1 for r in results_m23 if 'leadsub_d' in r)}")

Warmth list: removed 2 competence-overlapping words: ['intelligent', 'brilliant']
Warmth list final: 18 words

Leadership terms    : 16
Subordination terms : 16

State   Period         CompWarm d    ±std  LeadSub d    ±std
--------------------------------------------------------------


INFO:gensim.models.word2vec:collecting all words and their counts
INFO:gensim.models.word2vec:PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
INFO:gensim.models.word2vec:collected 1175 word types from a corpus of 3155 raw words and 100 sentences
INFO:gensim.models.word2vec:Creating a fresh vocabulary
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'effective_min_count=1 retains 1175 unique words (100.00% of original 1175, drops 0)', 'datetime': '2026-06-19T05:14:06.656474', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'prepare_vocab'}
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'effective_min_count=1 leaves 3155 word corpus (100.00% of original 3155, drops 0)', 'datetime': '2026-06-19T05:14:06.657503', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.

CA      hist_1920s                     --     -0.971   0.045


INFO:gensim.models.word2vec:EPOCH 11: training on 5079 raw words (3999 effective words) took 0.0s, 490386 effective words/s
INFO:gensim.models.word2vec:EPOCH 12: training on 5079 raw words (4001 effective words) took 0.0s, 450380 effective words/s
INFO:gensim.models.word2vec:EPOCH 13: training on 5079 raw words (4003 effective words) took 0.0s, 447228 effective words/s
INFO:gensim.models.word2vec:EPOCH 14: training on 5079 raw words (4034 effective words) took 0.0s, 557167 effective words/s
INFO:gensim.models.word2vec:EPOCH 15: training on 5079 raw words (4028 effective words) took 0.0s, 503670 effective words/s
INFO:gensim.models.word2vec:EPOCH 16: training on 5079 raw words (4041 effective words) took 0.0s, 462812 effective words/s
INFO:gensim.models.word2vec:EPOCH 17: training on 5079 raw words (4039 effective words) took 0.0s, 427471 effective words/s
INFO:gensim.models.word2vec:EPOCH 18: training on 5079 raw words (4006 effective words) took 0.0s, 463561 effective words/s
INFO:gen

CA      modern_2024        -0.497   0.182      0.508   0.118


...

OH      hist_1920s         -0.165   0.127                 --


INFO:gensim.models.word2vec:sample=0.001 downsamples 50 most-common words
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'downsampling leaves estimated 3120.115253501601 word corpus (78.4%% of prior 3981)', 'datetime': '2026-06-19T05:15:46.456508', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'prepare_vocab'}
INFO:gensim.models.word2vec:estimated required memory for 1354 words and 100 dimensions: 1760200 bytes
INFO:gensim.models.word2vec:resetting layer weights
INFO:gensim.utils:Word2Vec lifecycle event {'update': False, 'trim_rule': 'None', 'datetime': '2026-06-19T05:15:46.470094', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'build_vocab'}
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'training model with 1 workers on 1354 vocabulary and 100 f

WY      hist_1920s                     --     -0.879   0.193


INFO:gensim.models.word2vec:EPOCH 15: training on 5662 raw words (4441 effective words) took 0.0s, 790312 effective words/s
INFO:gensim.models.word2vec:EPOCH 16: training on 5662 raw words (4428 effective words) took 0.0s, 651627 effective words/s
INFO:gensim.models.word2vec:EPOCH 17: training on 5662 raw words (4448 effective words) took 0.0s, 782176 effective words/s
INFO:gensim.models.word2vec:EPOCH 18: training on 5662 raw words (4417 effective words) took 0.0s, 755714 effective words/s
INFO:gensim.models.word2vec:EPOCH 19: training on 5662 raw words (4425 effective words) took 0.0s, 801427 effective words/s
INFO:gensim.utils:Word2Vec lifecycle event {'msg': 'training on 113240 raw words (88845 effective words) took 0.2s, 461078 effective words/s', 'datetime': '2026-06-19T05:16:21.770809', 'gensim': '4.4.0', 'python': '3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'train'}
INFO:gensim.utils:Word2

WY      modern_2020                    --     -0.182   0.683

Cells scored (either measure): 9
Competence–Warmth cells : 5
Leadership–Subord cells : 7


In [11]:
m23_df = pd.DataFrame(results_m23).sort_values(['period', 'state'])

# Reliability flags per measure
m23_df['compwarm_reliable'] = (m23_df.get('compwarm_std', np.nan) < 0.25)
m23_df['leadsub_reliable']  = (m23_df.get('leadsub_std', np.nan) < 0.25)

OUT_M23 = OUTPUT_DIR / "measures23_scores_word2vec.csv"
m23_df.to_csv(OUT_M23, index=False)

print(f"Cells saved: {len(m23_df)}")
print(f"Reliable Competence–Warmth cells : {m23_df['compwarm_reliable'].sum()}")
print(f"Reliable Leadership–Subord cells : {m23_df['leadsub_reliable'].sum()}")
print(f"Saved to: {OUT_M23}")

# Curation documentation
curation_doc = {
    'measure2_competence_warmth': {
        'competence_list': 'garg_2018/adjectives_intelligencegeneral.txt (28 words)',
        'warmth_list': 'garg_2018/adjectives_sensitive.txt minus overlaps (18 words)',
        'overlap_removed': OVERLAP_REMOVED,
        'targets': 'garg_2018 male_pairs/female_pairs'
    },
    'measure3_leadership_subordination': {
        'leadership_list': 'custom_leadership/leadership_terms.txt (16 words)',
        'subordination_list': 'custom_leadership/subordination_terms.txt (16 words)',
        'curation_doc': 'word_lists/custom_leadership/README.md',
        'targets': 'garg_2018 male_pairs/female_pairs'
    },
    'sign_convention': 'positive = male more associated with first attribute (competence / leadership)'
}
with open(OUTPUT_DIR / "measures23_curation.json", 'w') as f:
    json.dump(curation_doc, f, indent=2)
print(f"Curation doc saved.")

Cells saved: 9
Reliable Competence–Warmth cells : 4
Reliable Leadership–Subord cells : 5
Saved to: D:\Shikha\bias_project\outputs\embeddings\word2vec\measures23_scores_word2vec.csv
Curation doc saved.


In [12]:
from collections import Counter
import pandas as pd

ERA_DIR = BASE_DIR / "word_lists" / "era_names"
  # word_lists/era_names ; if OUTPUT_DIR was reassigned, hardcode the path

# Map corpus periods -> era-list labels we built
PERIOD_TO_ERA = {
    'hist_1920s':   'historical_1920s',
    'modern_2020':  'modern_2024',
    'modern_2024':  'modern_2024',
}

# Load era name lists: era_label -> {'male': set, 'female': set}
era_names = {}
for f in ERA_DIR.glob("*_names_*.txt"):
    sex, _, era = f.stem.split("_", 2)
    era_names.setdefault(era, {})[sex] = set(f.read_text(encoding="utf-8").split())

# Coverage over the SAME tokenized cells Word2Vec scored on
rows = []
for (state, period), sentences in state_period_corpus.items():
    era = PERIOD_TO_ERA.get(period)
    if era not in era_names:
        continue
    counts = Counter(tok for sent in sentences for tok in sent)  # already simple_preprocess'd
    n_tokens = sum(counts.values())
    for sex in ("male", "female"):
        names   = era_names[era][sex]
        present = {n for n in names if counts[n] > 0}
        hits    = sum(counts[n] for n in present)
        rows.append({
            "state": state, "period": period, "sex": sex,
            "names_total":   len(names),
            "names_present": len(present),
            "coverage_pct":  round(100 * len(present) / len(names), 1) if names else 0,
            "name_hits":     hits,
            "corpus_tokens": n_tokens,
            "hits_per_10k":  round(10000 * hits / n_tokens, 2) if n_tokens else 0,
        })

cov = pd.DataFrame(rows)

print("=== Mean coverage by period × sex ===")
print(cov.groupby(["period", "sex"])[["names_present", "coverage_pct", "hits_per_10k"]].mean().round(2))

print("\n=== Cells with ANY name coverage ===")
print(f"{(cov['names_present'] > 0).sum()} of {len(cov)} (state,period,sex) groups have ≥1 name present")

print("\nLowest-coverage cells:")
print(cov.sort_values("coverage_pct").head(10).to_string(index=False))

VAL_DIR = BASE_DIR / "outputs" / "validation"
VAL_DIR.mkdir(parents=True, exist_ok=True)
cov.to_csv(VAL_DIR / "era_name_coverage.csv", index=False)
print(f"\nSaved: {VAL_DIR / 'era_name_coverage.csv'}")

=== Mean coverage by period × sex ===
                    names_present  coverage_pct  hits_per_10k
period      sex                                              
hist_1920s  female           2.44         12.22          6.44
            male             7.44         37.22         29.86
modern_2020 female           0.54          2.70          2.03
            male             1.50          7.50          4.20
modern_2024 female           2.92         14.62          3.70
            male             7.38         36.92         16.25

=== Cells with ANY name coverage ===
158 of 222 (state,period,sex) groups have ≥1 name present

Lowest-coverage cells:
state      period    sex  names_total  names_present  coverage_pct  name_hits  corpus_tokens  hits_per_10k
   CA modern_2020 female           20              0           0.0          0           5079           0.0
   CA modern_2020   male           20              0           0.0          0           5079           0.0
   CO modern_2020 female 

In [16]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(r"D:\Shikha\bias_project")
EMB = BASE_DIR / "outputs" / "embeddings"

files = {
    "word2vec": EMB / "word2vec" / "weat6_scores_word2vec_v2.csv",
    "glove":    EMB / "glove"    / "weat6_scores_glove.csv",
    "bert":     EMB / "bert"     / "seat_scores_bert.csv",
    "roberta":  EMB / "roberta"  / "seat_scores_roberta.csv",
}

for name, path in files.items():
    if path.exists():
        df = pd.read_csv(path)
        print(f"\n{name}  ({path.name}, {len(df)} rows)")
        print("  columns:", list(df.columns))
        print(df.head(2).to_string(index=False))
    else:
        print(f"\n{name}: FILE NOT FOUND at {path}")


word2vec  (weat6_scores_word2vec_v2.csv, 38 rows)
  columns: ['state', 'period', 'n_sentences', 'weat6_pairs_d', 'weat6_pairs_std', 'cov_male', 'cov_female', 'cov_career', 'cov_family', 'n_seeds', 'method', 'reliable']
state     period  n_sentences  weat6_pairs_d  weat6_pairs_std  cov_male  cov_female  cov_career  cov_family  n_seeds                    method  reliable
   CA hist_1920s          150         0.1262           0.2383         8           4           2           3        5 word2vec_pairs_effectsize      True
   MS hist_1920s          150        -0.5391           0.1964        10           7           4           4        5 word2vec_pairs_effectsize      True

glove  (weat6_scores_glove.csv, 50 rows)
  columns: ['state', 'text_chars', 'weat6_score', 'method', 'period']
state  text_chars  weat6_score         method     period
   AK      188041       0.1703 glove_weighted mixed_demo
   AL       63233       0.2114 glove_weighted mixed_demo

bert  (seat_scores_bert.csv, 50 rows)